# 🗐 Inter - Extração de Dados

> Melhorias: Fazer com que, de alguma maneira, Saiba se é Exposição Bruta ou Perdas Esperadas

In [134]:
import pdfplumber
import pandas as pd
import re
import numpy as np
from datetime import datetime

In [135]:
# escolha se é PE ou Exposição Bruta:
# PE = d. Análise da movimentação das perdas esperadas por estágio: (pág 42)
# Exposição Bruta = c. Análise da movimentação dos empréstimos e adiantamentos a clientes por estágio (pág 41)

In [136]:
with pdfplumber.open("Demonstrações Financeiras IFRS 2T25.pdf") as pdf:
    page = pdf.pages[41]   # página 42 (índice começa em 0)
    text = page.extract_text()

# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("d. Análise da movimentação das perdas esperadas por estágio")
end = text.find("Consolidado", start) # se pegar só total, conseguimos apenas o primeiro estágio. 
    
trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()] 
    

d. Análise da movimentação das perdas esperadas por estágio:
(Consideram perdas esperadas com operações de crédito e compromissos a serem honrados)
Saldo inicial em Transferência para Transferência para Transferência do Transferência do Constituição/ Saldo final em Saldo final em
Estágio 1 Baixas para prejuízo
01/01/2025 estágio 2 estágio 3 estágio 2 estágio 3 (reversão) 30/06/2025 31/12/2024
Cartão de crédito 427.310 (203.643) (1.582) 80.095 — — 324.029 626.209 427.310
Imobiliário 61.494 (72.215) (1.658) 11.198 45 — 51.360 50.224 61.494
Pessoal 81.172 (82.521) (28.923) 12.331 15.708 — 123.877 121.644 81.172
Empresas 10.640 (9.124) (559) 150 — — 18.241 19.348 10.640
Rural 6.993 (335) (119) — — — (3.046) 3.493 6.993
Total 587.609 (367.838) (32.841) 103.774 15.753 — 514.461 820.918 587.609
Saldo inicial em Transferência para Transferência para Transferência do Transferência do Constituição/ Saldo final em Saldo final em
Estágio 2 Baixas para prejuízo
01/01/2025 estágio 1 estágio 3 estági

In [137]:
PRODUTOS = [
    "Cartão de crédito",
    "Imobiliário",
    "Pessoal",
    "Empresas",
    "Rural",
]

def eh_produto(linha: str) -> bool:
    return any(linha.startswith(p) for p in PRODUTOS)

def reduzir_linha_produto(linha: str) -> str:
    """
    Mantém:
    - nome do produto
    - os dois últimos elementos da linha (número ou —)
    """
    partes = linha.split()

    # últimos dois elementos SEMPRE preservados
    ultimos_dois = partes[-2:]

    # nome do produto pode ter mais de uma palavra
    for produto in PRODUTOS:
        if linha.startswith(produto):
            return f"{produto} {' '.join(ultimos_dois)}"

    return linha  # fallback (não deveria acontecer)

def reduzir_texto(texto: str) -> str:
    linhas = [l.strip() for l in texto.splitlines() if l.strip()]
    saida = []

    for linha in linhas:
        # Regra 2
        if linha.startswith("Total"):
            continue

        # Regra 3
        if linha.lower().startswith("saldo inicial"):
            continue

        # Regra 4 + 1
        if eh_produto(linha):
            saida.append(reduzir_linha_produto(linha))
        else:
            # Mantém cabeçalhos, títulos, estágios, datas etc
            saida.append(linha)

    return "\n".join(saida)


In [138]:
texto_reduzido = reduzir_texto(trecho)
print(texto_reduzido)


d. Análise da movimentação das perdas esperadas por estágio:
(Consideram perdas esperadas com operações de crédito e compromissos a serem honrados)
Estágio 1 Baixas para prejuízo
01/01/2025 estágio 2 estágio 3 estágio 2 estágio 3 (reversão) 30/06/2025 31/12/2024
Cartão de crédito 626.209 427.310
Imobiliário 50.224 61.494
Pessoal 121.644 81.172
Empresas 19.348 10.640
Rural 3.493 6.993
Estágio 2 Baixas para prejuízo
01/01/2025 estágio 1 estágio 3 estágio 1 estágio 3 (reversão) 30/06/2025 31/12/2024
Cartão de crédito 193.762 172.247
Imobiliário 33.259 49.709
Pessoal 34.764 56.509
Empresas 3.579 4.670
Rural — —
Estágio 3 Baixas para prejuízo
01/01/2025 estágio 1 estágio 2 estágio 1 estágio 2 (reversão) 30/06/2025 31/12/2024
Cartão de crédito 958.882 970.797
Imobiliário 87.758 66.626
Pessoal 467.857 441.441
Empresas 39.019 17.276
Rural 1.280 (1)


In [139]:
# Extração de anos e meses

def trimestre_from_date(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    trimestre_map = {1: "1T",3: "1T", 6: "2T", 9: "3T", 12: "4T"} # melhorar parte do código
    trimestre = trimestre_map.get(dt.month)

    if not trimestre:
        raise ValueError(f"Mês inesperado na data: {date_str}")

    return f"{trimestre}{str(dt.year)[-2:]}"

def extract_anos(texto: str) -> list[str]:
    datas = re.findall(r"\d{2}/\d{2}/\d{4}", texto)

    if len(datas) < 2:
        raise ValueError("Não foi possível encontrar duas datas finais")

    datas_finais = datas[-2:]
    return [trimestre_from_date(d) for d in datas_finais]


In [140]:
extract_anos(texto_reduzido)
# ['3T25', '4T24']


['2T25', '4T24']

In [141]:
# Separa em blocos

def split_blocos_estagio(texto: str) -> list[tuple[int, list[str]]]:
    linhas = texto.splitlines()

    blocos = []
    estagio_atual = None
    buffer = []

    for linha in linhas:
        m = re.match(r"\s*Estágio\s+(\d+)", linha)
        if m:
            if estagio_atual is not None:
                blocos.append((estagio_atual, buffer))
                buffer = []
            estagio_atual = int(m.group(1))
            continue

        if estagio_atual is not None:
            buffer.append(linha)

    if estagio_atual is not None:
        blocos.append((estagio_atual, buffer))

    return blocos


In [142]:
split_blocos_estagio(texto_reduzido)

[(1,
  ['01/01/2025 estágio 2 estágio 3 estágio 2 estágio 3 (reversão) 30/06/2025 31/12/2024',
   'Cartão de crédito 626.209 427.310',
   'Imobiliário 50.224 61.494',
   'Pessoal 121.644 81.172',
   'Empresas 19.348 10.640',
   'Rural 3.493 6.993']),
 (2,
  ['01/01/2025 estágio 1 estágio 3 estágio 1 estágio 3 (reversão) 30/06/2025 31/12/2024',
   'Cartão de crédito 193.762 172.247',
   'Imobiliário 33.259 49.709',
   'Pessoal 34.764 56.509',
   'Empresas 3.579 4.670',
   'Rural — —']),
 (3,
  ['01/01/2025 estágio 1 estágio 2 estágio 1 estágio 2 (reversão) 30/06/2025 31/12/2024',
   'Cartão de crédito 958.882 970.797',
   'Imobiliário 87.758 66.626',
   'Pessoal 467.857 441.441',
   'Empresas 39.019 17.276',
   'Rural 1.280 (1)'])]

In [143]:
def parse_linhas_produto(linhas: list[str]) -> list[dict]:
    rows = []

    for linha in linhas:
        for produto in PRODUTOS:
            if linha.startswith(produto):
                partes = linha.replace(produto, "").strip().split()
                v1, v2 = partes

                rows.append({
                    "produto": produto,
                    "v1": v1,
                    "v2": v2,
                })

    return rows


In [144]:
def normalizar_valor(valor: str):
    
    if valor is None:
        return None
    
    valor = valor.strip()
    
    if valor in ["—", "-", ""]:
        return None
    
    # Remove espaços internos
    valor = valor.replace(" ", "")
    
    # Detecta número negativo no formato (123) ou (17.046)
    if valor.startswith("(") and valor.endswith(")"):
        valor = "-" + valor[1:-1]
    
    # Remove separador de milhar
    valor = valor.replace(".", "")
    
    # Se existir vírgula decimal (caso futuro)
    valor = valor.replace(",", ".")
    
    return float(valor)


In [145]:
def construir_linhas(texto_reduzido: str) -> list[dict]:
    anos = extract_anos(texto_reduzido)
    blocos = split_blocos_estagio(texto_reduzido)

    linhas_finais = []

    for estagio, linhas_bloco in blocos:
        produtos = parse_linhas_produto(linhas_bloco)

        for ano, chave in zip(anos, ["v1", "v2"]):
            for p in produtos:
                linhas_finais.append({
                    "ano": ano,
                    "banco": "Inter",
                    "produto": p["produto"],
                    "PD": "-",
                    "Estágio": estagio,
                    "Perda Esperada": normalizar_valor(p[chave]),
                })

    return linhas_finais


In [146]:
df = pd.DataFrame(construir_linhas(texto_reduzido))
df

,ano,banco,produto,PD,Estágio,Perda Esperada
0,2T25,Inter,Cartão de crédito,-,1,626209.0
1,2T25,Inter,Imobiliário,-,1,50224.0
2,2T25,Inter,Pessoal,-,1,121644.0
3,2T25,Inter,Empresas,-,1,19348.0
4,2T25,Inter,Rural,-,1,3493.0
5,4T24,Inter,Cartão de crédito,-,1,427310.0
6,4T24,Inter,Imobiliário,-,1,61494.0
7,4T24,Inter,Pessoal,-,1,81172.0
8,4T24,Inter,Empresas,-,1,10640.0
9,4T24,Inter,Rural,-,1,6993.0


### 📊 Passar para o Excel

In [147]:
import openpyxl
from openpyxl import load_workbook

In [148]:
# Carrega arquivos
dados_bancarios_xlsx = load_workbook("Excel - Dados bancários.xlsx")
inter_sheet = dados_bancarios_xlsx["Inter"]

# Verifica se existe todos os anos ( se não, adiciona +1 coluna a direita )
anos_df = df["ano"].unique()

for ano_df in anos_df:
    
    existe = False
    
    for col in range(1, inter_sheet.max_column + 1):
        if str(inter_sheet.cell(row=5, column=col).value).strip() == str(ano_df):
            existe = True
            break
    
    if not existe:
        nova_coluna = inter_sheet.max_column + 1
        inter_sheet.cell(row=5, column=nova_coluna, value=ano_df)

# Identificação da coluna Tipo
ultima_coluna_df = df.columns[-1]

if ultima_coluna_df not in ["Exposição Bruta", "Perda Esperada"]:
    raise ValueError("Última coluna do DataFrame não é métrica válida.")

# Preenchimento dos dados
for _, row in df.iterrows():
    
    tipo_df = ultima_coluna_df
    estagio_df = f"Estágio {row['Estágio']}"
    produto_df = row["produto"]
    ano_df = row["ano"]
    valor_df = row[ultima_coluna_df]
    
    linha_encontrada = None
    coluna_encontrada = None
    
    # 🔎 Encontrar linha correta (Tipo + Estágio + Produto)
    for r in range(1, inter_sheet.max_row + 1):
        
        tipo_excel = str(inter_sheet.cell(row=r, column=2).value).strip()
        estagio_excel = str(inter_sheet.cell(row=r, column=3).value).strip()
        produto_excel = str(inter_sheet.cell(row=r, column=4).value).strip()
        
        if (
            tipo_excel == tipo_df and
            estagio_excel == estagio_df and
            produto_excel == produto_df
        ):
            linha_encontrada = r
            break
    
    # 🔎 Encontrar coluna correta (Ano na linha 5)
    for c in range(1, inter_sheet.max_column + 1):
        if str(inter_sheet.cell(row=5, column=c).value).strip() == str(ano_df):
            coluna_encontrada = c
            break
    
    # 🔥 Preencher célula
    if linha_encontrada and coluna_encontrada:
        inter_sheet.cell(
            row=linha_encontrada,
            column=coluna_encontrada,
            value=valor_df
        )

# Salva arquivo
dados_bancarios_xlsx.save("Excel - Dados bancários.xlsx")
